# Homework — Special Methods, Access Modifiers, Inheritance & Files

**Theme: Gym Membership System**

This homework mirrors what we did in class, applied to a new scenario, so you can check your own
understanding independently. Sections build on each other — do them in order.

| Section | Topics |
|---|---|
| 🟢 Core | `__init__`, access modifiers, `__str__`, comparison operators |
| 🟡 Stretch | Inheritance, `super()`, overriding, composition |
| 🔴 Challenge | File I/O, iteration, an extra special method |

As in class: each class lives in one cell. If a later exercise asks you to add a method to a
class you already wrote, scroll up, add it there, and re-run.


## 🟢 Core — Exercise 1: The `Member` class

Every gym member is identified by an automatically generated, incrementing membership ID
(the first member created gets ID 1, the second gets ID 2, etc. — use a **class attribute** as a
counter).

Build a `Member` class with:

**Instance attributes**
- `name` → **public**
- `email` → **public**
- `age` → **protected**
- `member_id` → **private**, auto-generated in `__init__` (don't take it as a parameter)

**Instance methods**
- `get_member_id()` — returns the member's ID
- `__str__` — e.g. `"Member #1: Alice (alice@mail.com)"`
- `__eq__` — two members are equal if they have the same `member_id`
- `__lt__` — one member is "less than" another if they are **younger**


In [1]:
from __future__ import annotations

In [53]:
class Member:
    _id_counter = 0   # class attribute: shared counter for auto-generating IDs

    def __init__(self, name: str, email: str, age: int):
        self.name = name
        self.email = email
        # TODO: store age as PROTECTED
        self._age = age
        # TODO: generate and store a PRIVATE member_id using Member._id_counter
        self.__member_id = Member._id_counter
        #       (increment the counter each time a new Member is created)
        Member._id_counter+=1

    def get_member_id(self) -> int:
        # TODO
        return self.__member_id

    def get_member_age(self) -> int:
        # TODO
        return self._age

    def __str__(self) -> str:
        # TODO
        return f"Member #{self.__member_id}: {self.name} ({self.email})"

    def __eq__(self, other: object) -> bool:
        # TODO
        if not isinstance(other, Member):
            raise TypeError(f"other object should be of type 'class Member', encountered {type(other)}")
        return self.__member_id == other.__member_id

    def __lt__(self, other: Member) -> bool:
        # TODO
        return self._age < other._age


In [54]:
alice = Member("Alice", "alice@mail.com", 28)
bob = Member("Bob", "bob@mail.com", 34)

print(alice)                     # Member #1: Alice (alice@mail.com)
print(bob)                       # Member #2: Bob (bob@mail.com)
print(alice.get_member_id())     # 1
print(alice < bob)               # True (Alice is younger)
print(alice == bob)              # False


Member #0: Alice (alice@mail.com)
Member #1: Bob (bob@mail.com)
0
True
False


## 🟡 Stretch — Exercise 2: `BasicMember` and `PremiumMember`

Create two subclasses of `Member`:

**`BasicMember`**
- No extra attributes.
- `get_monthly_fee()` — always returns `200`.

**`PremiumMember`**
- Extra attribute: `personal_trainer` (a name, e.g. `"Coach Kim"`).
- `get_monthly_fee()` — **overrides** the basic fee: calls `super()`'s fee somehow is not possible
  here since `Member` itself has no fee — instead, `PremiumMember.get_monthly_fee()` should
  return `500` (base) `+ 150` if `personal_trainer` is set, otherwise just `500`.

Use `super().__init__(...)` in both subclasses to set up the inherited attributes.


In [55]:
class BasicMember(Member):
    def __init__(self, name: str, email: str, age: int):
        # TODO: call the parent constructor with super()
        super().__init__(name, email, age)

    def get_monthly_fee(self) -> float:
        # TODO: always 200
        return 200


class PremiumMember(Member):
    def __init__(self, name: str, email: str, age: int, personal_trainer: str | None):
        # TODO: call the parent constructor with super()
        super().__init__(name, email, age)
        # TODO: store personal_trainer
        self.personal_trainer = personal_trainer

    def get_monthly_fee(self) -> float:
        # TODO: 500, plus 150 if personal_trainer is set
        return 500+150 if self.personal_trainer else 500


In [56]:
basic = BasicMember("Chris", "chris@mail.com", 22)
premium = PremiumMember("Dana", "dana@mail.com", 40, "Coach Kim")

print(basic.get_monthly_fee())     # 200
print(premium.get_monthly_fee())   # 650
print(basic)                       # inherited __str__ still works


200
650
Member #2: Chris (chris@mail.com)


## 🟡 Stretch — Exercise 3: The `Gym` class

Create a `Gym` class that **stores** `Member` objects (composition):

- `__init__(self, name)` — stores the gym's name and an empty list of members.
- `add_member(member)` — adds a member.
- `total_monthly_revenue()` — sums `get_monthly_fee()` across all members.
- `find_by_email(email)` — returns the member with that email, or `None` if not found.
- `__len__` — number of members.
- `__getitem__(index)` — index into the members list.


In [114]:
from pathlib import Path


class Gym:
    def __init__(self, name: str):
        self.name = name
        # TODO: initialise an empty list of members
        self.members: list[Member] = []

    def add_member(self, member: Member) -> None:
        # TODO
        self.members.append(member)

    def total_monthly_revenue(self) -> float:
        # TODO
        return sum(member.get_monthly_fee() for member in self.members)

    def find_by_email(self, email:str) -> Member|None:
        # TODO
        return next((member for member in self.members if member.email==email), None)

    def __len__(self) -> int:
        # TODO
        return len(self.members)

    def __getitem__(self, index: int) -> Member:
        # TODO
        return self.members[index]

    def __iter__(self) -> Gym:
        self.next_id = 0
        return self

    def __next__(self) -> Member:
        try: 
            next_member = self.members[self.next_id]
            self.next_id+=1
            return next_member
        except IndexError:
            raise StopIteration ("Members exausted")
        

    def __bool__(self) -> bool:

        return len(self) > 0

    def save_to_file(self, file_name: Path) -> None:  
        with open(file_name, "w") as file:
            file.writelines(f"{member.name}, {member.email}, {member.get_member_age()}\n" for member in self.members)

    @classmethod
    def load_from_file(cls, file_name: Path, name: str) -> Gym:  
        new_gym = cls(name)
        with open(file_name, "r") as file:
            for line in file:
                name, email, age = line.strip().split(", ")
                new_gym.add_member(Member(name=name,email=email,age=int(age)))

        return new_gym



In [115]:
gym = Gym("PowerGym")
gym.add_member(basic)
gym.add_member(premium)

print(len(gym))                        # 2
print(gym.total_monthly_revenue())     # 200 + 650 = 850
print(gym.find_by_email("dana@mail.com"))


2
850
Member #3: Dana (dana@mail.com)


In [116]:
print(Member._id_counter)

20


In [117]:
print(gym)

In [118]:
print(gym.members)
print(gym.members[0])
print(gym.members[1])

[<__main__.BasicMember object at 0x00000209FB214E10>, <__main__.PremiumMember object at 0x00000209FB2171D0>]
Member #2: Chris (chris@mail.com)
Member #3: Dana (dana@mail.com)


## 🔴 Challenge — Exercise 4: Make `Gym` iterable and "truthy"

Add to your `Gym` class:
- `__iter__` / `__next__` — so `for member in gym:` works directly.
- `__bool__` — a `Gym` should be considered `True` if it has at least one member, `False`
  otherwise (hint: this can be a one-liner using `len(self)`).


In [119]:
for member in gym:
    print(member)

empty_gym = Gym("New Gym")
print(bool(gym))         # True
print(bool(empty_gym))   # False

print(gym)         
print(empty_gym)

Member #2: Chris (chris@mail.com)
Member #3: Dana (dana@mail.com)
True
False


## 🔴 Challenge — Exercise 5: Save and load members

Add to `Gym`:
- `save_to_file(filename)` — writes one line per member as `name,email,age`, using a context
  manager. (Hint: you'll need a way to read the protected `age` — add a small `get_age()` method
  to `Member` if you don't have one yet.)
- `load_from_file(filename, name)` — a **class method** that reads the file back and returns a
  new `Gym` of plain `Member` objects (member IDs will simply be re-assigned in load order,
  that's OK).


In [121]:
gym.save_to_file(Path("members.txt"))
reloaded_gym = Gym.load_from_file(Path("members.txt"), "Reloaded Gym")
for member in reloaded_gym:
    print(member)


Member #22: Chris (chris@mail.com)
Member #23: Dana (dana@mail.com)
